# Assignment Case B — Vendor Performance
## Working with PySpark DataFrames for Analytics
**Day 42 | Batch 13 | 03 Mei 2026**

---

| Field | Isi |
|-------|-----|
| **Nama** | *Muhammad Qois Huzyan Octava* |
| **NIM** | *(isi NIM kamu)* |
| **Case** | Case B — Procurement & Vendor Performance |
| **Source** | Hive: `adventureworks.*` |
| **Target** | Hive: `adventureworks_curated.fact_vendor_performance` |

---

## ⚠️ Filter Penting

> **Kamu HANYA boleh menggunakan data dengan ShipMethod: `CARGO TRANSPORT 5` dan `OVERNIGHT J-FAST`.**
> Filter ini harus diterapkan sebelum agregasi.

## 🎯 Business Question

1. **Vendor mana yang paling on-time** dalam pengiriman via CARGO TRANSPORT 5 dan OVERNIGHT J-FAST?
2. **Apakah harga beli dari vendor sudah kompetitif** dibanding StandardCost produk?
3. **Vendor mana yang memiliki VendorScore tertinggi** secara keseluruhan?
4. **Korelasi** antara CreditRating vendor dan OnTimeRate?

## 📐 Formula VendorScore

```
VendorScore = (OnTimeRate * 0.6) + ((100 - ABS(AvgPriceVariance)) * 0.4)
```

- **OnTimeRate** = OnTimeOrders / TotalOrders * 100  (ShipDate <= DueDate)
- **AvgPriceVariance** = rata-rata (UnitPrice - StandardCost) per baris PO detail
- ⚠️ Exclude baris dengan ShipDate IS NULL dari kalkulasi OnTimeRate

## 📋 Kolom Target: `fact_vendor_performance`

| Kolom | Tipe | Keterangan |
|-------|------|------------|
| `vendor_id` | INT | ID vendor |
| `vendor_name` | STRING | Nama vendor |
| `credit_rating` | INT | Rating kredit (1-5) |
| `ship_method` | STRING | Metode pengiriman |
| `total_orders` | INT | Total PO |
| `on_time_orders` | INT | PO dengan ShipDate <= DueDate |
| `on_time_rate` | DECIMAL(5,2) | % ketepatan pengiriman |
| `avg_lead_time_days` | DECIMAL(5,1) | Rata-rata (ShipDate - OrderDate) |
| `avg_price_variance` | DECIMAL(10,2) | Rata-rata (UnitPrice - StandardCost) |
| `vendor_score` | DECIMAL(5,2) | Formula scorecard |
| `load_timestamp` | TIMESTAMP | Waktu load |


## 0. Persiapan: Buat Hive External Tables untuk Vendor Performance

Sebelum mulai analisis, kamu perlu menyiapkan tabel sumber sebagai
**Hive External Tables**. Pola tutorialnya sama dengan:
- `02_adventureworks_to_hdfs.ipynb` → extract PostgreSQL ke HDFS Parquet (dimensi + fakta dengan partisi)
- `03_hdfs_to_hive.ipynb` → buat External Tables di atas Parquet

**Tabel yang akan dibuat di Hive (`adventureworks.*`):**
| Hive Table | Sumber PostgreSQL | Catatan |
|---|---|---|
| `fact_purchase_orders` | `Purchasing.PurchaseOrderHeader` | partisi `order_year`, `order_month` |
| `fact_purchase_details` | `Purchasing.PurchaseOrderDetail` | tanpa partisi |
| `dim_vendor` | `Purchasing.Vendor` | |
| `dim_ship_method` | `Purchasing.ShipMethod` | |
| `dim_product` | `Production.Product` | |

> **Lewati bagian ini** jika tabel sudah ada di Hive.

### 0.1 Setup SparkSession JDBC (untuk Extract ke HDFS)

Mirip dengan `02_adventureworks_to_hdfs.ipynb`. SparkSession ini
**tanpa Hive support** — hanya untuk extract data dari PostgreSQL ke HDFS sebagai Parquet.

In [2]:
# TODO: setup environment & import
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /home/jovyan/work/jars/postgresql-42.7.3.jar pyspark-shell'  # TODO: '--jars /home/jovyan/work/jars/postgresql-42.7.3.jar pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F_jdbc

# TODO: buat spark_jdbc tanpa Hive support
spark_jdbc = SparkSession.builder \
    .master('local[*]') \
    .appName('Vendor-to-HDFS') \
    .config('spark.sql.parquet.compression.codec', 'snappy') \
    .getOrCreate()

# TODO: konfigurasi JDBC PostgreSQL AdventureWorks
JDBC_URL   = 'jdbc:postgresql://adventureworks-postgres:5432/postgres'  # TODO: 'jdbc:postgresql://adventureworks-postgres:5432/postgres'
JDBC_PROPS = {
                    "user": "postgres",
                    "password": "My_password1",
                    "driver": "org.postgresql.Driver"
                }  # TODO: dict berisi 'user', 'password', 'driver'
HDFS_BASE  = 'hdfs://namenode:9000/datalake/raw'  # TODO: 'hdfs://namenode:9000/datalake/raw'

def read_pg(schema, table):
    return spark_jdbc.read.jdbc(
        url=JDBC_URL,
        table=f'"{schema}"."{table}"',
        properties=JDBC_PROPS
    )

print(f'spark_jdbc {spark_jdbc.version} ready')


spark_jdbc 3.5.0 ready


### 0.2 Extract Tabel Dimensi ke HDFS

Tabel dimensi tidak perlu di-partisi (ukurannya kecil).

In [7]:
# TODO: lengkapi list tabel dimensi
dim_tables = [
    ('Purchasing', 'Vendor'),  # TODO: ('Purchasing', 'Vendor')
    ('Purchasing', 'ShipMethod'),  # TODO: ('Purchasing', 'ShipMethod')
    ('Production', 'Product'),  # TODO: ('Production', 'Product')
    ('Purchasing', 'PurchaseOrderDetail')  # TODO: ('Purchasing', 'PurchaseOrderDetail')  -- treat as dim (tanpa partisi)
]

for schema, table in dim_tables:
    df = read_pg(schema, table)  # TODO: read_pg(schema, table)
    hdfs_path = f'{HDFS_BASE}/{schema}/{table}'  # TODO: f'{HDFS_BASE}/{schema}/{table}'
    df.write.mode('overwrite').parquet(hdfs_path)
    print(f'  {schema}.{table:30s} -> {df.count():6,} rows -> HDFS')

print('Tabel dimensi selesai disimpan!')


Py4JJavaError: An error occurred while calling o75.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 4.0 failed 1 times, most recent failure: Lost task 0.0 in stage 4.0 (TID 4) (spark-jupyter executor driver): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to hdfs://namenode:9000/datalake/raw/Purchasing/Vendor.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:774)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeTask(FileFormatWriter.scala:420)
	at org.apache.spark.sql.execution.datasources.WriteFilesExec.$anonfun$doExecuteWrite$1(WriteFiles.scala:100)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2(RDD.scala:890)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2$adapted(RDD.scala:890)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.rdd.RDD.compu

### 0.3 Extract Tabel Fakta dengan Partisi

`PurchaseOrderHeader` berukuran besar dan punya kolom `OrderDate`. Partisi
berdasarkan `order_year` dan `order_month` — sama polanya dengan
`SalesOrderHeader` di notebook `02`.

In [4]:
# TODO: read PurchaseOrderHeader dari PostgreSQL
df_poh = read_pg('Purchasing', 'PurchaseOrderHeader')  # TODO: read_pg('Purchasing', 'PurchaseOrderHeader')

# TODO: tambahkan kolom partisi order_year & order_month dari OrderDate
df_poh_part = df_poh \
    .withColumn('order_year',  F_jdbc.year('orderdate')) \
    .withColumn('order_month', F_jdbc.month('orderdate'))  # TODO: F_jdbc.month('orderdate')

po_path = f'{HDFS_BASE}/Purchasing/PurchaseOrderHeader'

# TODO: simpan ke HDFS dengan partisi (order_year, order_month)
df_poh_part.write.mode('overwrite') \
    .partitionBy('order_year', 'order_month') \
    .parquet(po_path)

print(f'  PurchaseOrderHeader -> {df_poh.count():,} rows -> HDFS (partisi order_year/order_month)')


Py4JJavaError: An error occurred while calling o51.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1.0 (TID 1) (spark-jupyter executor driver): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to hdfs://namenode:9000/datalake/raw/Purchasing/PurchaseOrderHeader.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:774)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeTask(FileFormatWriter.scala:420)
	at org.apache.spark.sql.execution.datasources.WriteFilesExec.$anonfun$doExecuteWrite$1(WriteFiles.scala:100)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2(RDD.scala:890)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2$adapted(RDD.scala:890)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.

### 0.4 Setup SparkSession dengan Hive Support

Mirip dengan `03_hdfs_to_hive.ipynb`. Tutup `spark_jdbc` dulu, lalu buat
SparkSession baru **dengan** Hive support untuk membuat External Tables.

In [ ]:
# TODO: stop spark_jdbc agar tidak konflik resource
spark_jdbc.stop()

from pyspark.sql import SparkSession

# TODO: buat spark_hive dengan Hive support
spark_hive = SparkSession.builder \
    .master('local[*]') \
    .appName('Vendor-HiveSetup') \
    .config('spark.jars', '/home/jovyan/work/jars/postgresql-42.7.3.jar') \
    .config('hive.metastore.uris', ___)  # TODO: 'thrift://hive-metastore:9083'
    .config('spark.sql.warehouse.dir', ___)  # TODO: 'hdfs://namenode:9000/user/hive/warehouse'
    ___  # TODO: .enableHiveSupport()
    .getOrCreate()

HDFS_BASE = 'hdfs://namenode:9000/datalake/raw'

# TODO: buat database adventureworks jika belum ada
spark_hive.sql(___)  # TODO: CREATE DATABASE IF NOT EXISTS adventureworks
spark_hive.sql('USE adventureworks')
print(f'spark_hive {spark_hive.version} ready')
spark_hive.sql('SHOW DATABASES').show()


### 0.5 Buat Hive External Tables — Tabel Dimensi

Tabel dimensi tanpa partisi → schema otomatis di-infer.

In [ ]:
# Helper function — sama persis dengan pola di 03_hdfs_to_hive.ipynb
def create_external_table(table_name, hdfs_path):
    spark_hive.sql(f'DROP TABLE IF EXISTS adventureworks.{table_name}')
    spark_hive.sql(f"""
        CREATE EXTERNAL TABLE adventureworks.{table_name}
        USING PARQUET
        LOCATION '{hdfs_path}'
    """)
    count = spark_hive.sql(f'SELECT COUNT(*) FROM adventureworks.{table_name}').collect()[0][0]
    print(f'  {table_name:35s} -> {count:6,} rows')

# TODO: lengkapi mapping nama tabel Hive -> path HDFS
hive_dim_tables = [
    ___,  # TODO: ('dim_vendor',         f'{HDFS_BASE}/Purchasing/Vendor')
    ___,  # TODO: ('dim_ship_method',    f'{HDFS_BASE}/Purchasing/ShipMethod')
    ___,  # TODO: ('dim_product',        f'{HDFS_BASE}/Production/Product')
    ___,  # TODO: ('fact_purchase_details', f'{HDFS_BASE}/Purchasing/PurchaseOrderDetail')
]

for tname, path in hive_dim_tables:
    create_external_table(___, ___)  # TODO

print('Tabel dimensi Hive selesai!')


### 0.6 Buat Hive External Table — Tabel Fakta dengan Partisi

Untuk tabel berpartisi, perlu deklarasi `PARTITIONED BY` di DDL
lalu jalankan `MSCK REPAIR TABLE` agar Hive mengenali partisinya.
Pola sama dengan `fact_sales_orders` di `03_hdfs_to_hive.ipynb`.

In [ ]:
po_path = f'{HDFS_BASE}/Purchasing/PurchaseOrderHeader'

# TODO: drop dulu kalau ada
spark_hive.sql(___)  # TODO: 'DROP TABLE IF EXISTS adventureworks.fact_purchase_orders'

# TODO: buat External Table dengan skema eksplisit + PARTITIONED BY
# Petunjuk: kolom partisi (order_year, order_month) JANGAN ditulis di list kolom utama
spark_hive.sql(f"""
    CREATE EXTERNAL TABLE adventureworks.fact_purchase_orders (
        PurchaseOrderID    INT,
        RevisionNumber     SMALLINT,
        Status             SMALLINT,
        EmployeeID         INT,
        VendorID           INT,
        ShipMethodID       INT,
        OrderDate          TIMESTAMP,
        ShipDate           TIMESTAMP,
        ___                ___,  -- TODO: tambahkan kolom DueDate, SubTotal, TaxAmt, Freight, TotalDue, ModifiedDate
        ModifiedDate       TIMESTAMP
    )
    PARTITIONED BY (___ INT, ___ INT)  -- TODO: order_year, order_month
    STORED AS PARQUET
    LOCATION '{po_path}'
""")

# TODO: jalankan MSCK REPAIR TABLE agar partisi terdeteksi
spark_hive.sql(___)  # TODO: 'MSCK REPAIR TABLE adventureworks.fact_purchase_orders'

cnt = spark_hive.sql('SELECT COUNT(*) FROM adventureworks.fact_purchase_orders').collect()[0][0]
print(f'  fact_purchase_orders -> {cnt:,} rows')

spark_hive.sql('SHOW TABLES IN adventureworks').show(20)
spark_hive.stop()
print('Setup Hive External Tables selesai.')


## 0. Setup SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('CaseB_VendorPerformance') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

print(f'Spark {spark.version} ready')
spark.sql('SHOW TABLES IN adventureworks').show(20)

## 1. Load Data dari Hive External Tables

Semua tabel sumber sudah ada di Hive sebagai External Tables (dibuat di Bagian 0).
Baca via `spark.table('adventureworks.<nama_tabel>')`.

In [ ]:
# TODO: Load semua tabel dari Hive
df_po_header  = ___  # TODO: spark.table('adventureworks.fact_purchase_orders')
df_po_detail  = ___  # TODO: spark.table('adventureworks.fact_purchase_details')
df_vendor     = ___  # TODO: spark.table('adventureworks.dim_vendor')
df_shipmethod = ___  # TODO: spark.table('adventureworks.dim_ship_method')
df_product    = ___  # TODO: spark.table('adventureworks.dim_product')

print('Row counts:')
for name, df in [
    ('fact_purchase_orders',  df_po_header),
    ('fact_purchase_details', df_po_detail),
    ('dim_vendor',            df_vendor),
    ('dim_ship_method',       df_shipmethod),
    ('dim_product',           df_product),
]:
    print(f'  {name}: {df.count():,}')


## 2. Eksplorasi Data

In [ ]:
# TODO: Eksplorasi
print('=== ShipMethod yang tersedia ===')
# TODO: tampilkan semua ShipMethod

print('\n=== Sample PurchaseOrderHeader ===')
# TODO: show(5)

print('\n=== Distribusi PO per ShipMethod ===')
# TODO: join header dengan shipmethod, hitung count per nama metode

## 3. Extract — Filter & Join

**⚠️ Terapkan filter ShipMethod di sini!**

In [ ]:
# TODO: Filter hanya ShipMethod CARGO TRANSPORT 5 dan OVERNIGHT J-FAST
VALID_METHODS = ['CARGO TRANSPORT 5', 'OVERNIGHT J-FAST']

# Step 1: Filter ShipMethod
df_methods_filtered = df_shipmethod.filter(F.col('Name').isin(VALID_METHODS))

# Step 2: Join PurchaseOrderHeader dengan ShipMethod yang sudah difilter
# (ini otomatis memfilter PO berdasarkan ShipMethod)
df_po_filtered = ___  # TODO: join header dengan methods_filtered

print(f'Total PO setelah filter: {df_po_filtered.count():,}')
df_po_filtered.groupBy('Name').count().show()

# Step 3: Join dengan PO Detail dan Product
df_enriched = ___  # TODO:
# df_po_filtered
#   JOIN df_po_detail ON PurchaseOrderID
#   JOIN df_vendor    ON VendorID == BusinessEntityID
#   JOIN df_product   ON ProductID (ambil StandardCost)

# Step 4: Tambahkan kolom kalkulasi:
# - is_on_time      = ShipDate <= DueDate AND ShipDate IS NOT NULL  (1 atau 0)
# - lead_time_days  = datediff(ShipDate, OrderDate)                  (hanya jika ShipDate not null)
# - price_variance  = UnitPrice - StandardCost

# TODO: withColumn is_on_time, lead_time_days, price_variance

print(f'Total rows enriched: {df_enriched.count():,}')
df_enriched.printSchema()

## 4. Transform — Vendor Summary per ShipMethod

**Grain: 1 baris per (vendor, ship_method)**

In [ ]:
# TODO: Hitung agregasi per (vendor, ship_method)
# Group by: VendorID, vendor_name, credit_rating, ship_method_name
# Agregasi:
#   - total_orders       = count(PurchaseOrderID)
#   - on_time_orders     = sum(is_on_time)
#   - on_time_rate       = ROUND(on_time_orders / total_orders * 100, 2)
#   - avg_lead_time_days = ROUND(avg(lead_time_days), 1)  ← exclude null ShipDate
#   - avg_price_variance = ROUND(avg(price_variance), 2)

df_vendor_summary = ___  # TODO

# Step 2: Hitung VendorScore
# VendorScore = (OnTimeRate * 0.6) + ((100 - ABS(AvgPriceVariance)) * 0.4)
df_vendor_summary = df_vendor_summary.withColumn(
    'vendor_score',
    F.round(
        ___  # TODO: formula vendor score
        , 2)
)

print(f'Total vendor summary rows: {df_vendor_summary.count():,}')
df_vendor_summary.orderBy(F.desc('vendor_score')).show(20, truncate=False)

## 5. Window Functions — Ranking Vendor per ShipMethod

In [ ]:
# TODO: Rank vendor per ship_method berdasarkan vendor_score
win_by_method = Window.partitionBy('ship_method').orderBy(F.desc('vendor_score'))

df_ranked = df_vendor_summary \
    .withColumn('rank_in_method', F.rank().over(win_by_method)) \
    .withColumn('score_pct_rank', F.percent_rank().over(win_by_method))

print('=== Top 5 Vendor per ShipMethod ===')
df_ranked.filter(F.col('rank_in_method') <= 5) \
         .orderBy('ship_method', 'rank_in_method') \
         .show(20, truncate=False)

# TODO: Overall rank (tanpa partisi ship_method)
# Aggregate: kalau satu vendor muncul di dua ship method, ambil avg vendor_score
df_overall = ___  # TODO: groupBy vendor_id, vendor_name → avg(vendor_score)
win_overall = Window.orderBy(F.desc('avg_vendor_score'))
df_overall_ranked = df_overall.withColumn('overall_rank', F.rank().over(win_overall))

print('\n=== Overall Vendor Ranking ===')
df_overall_ranked.show(20, truncate=False)

## 6. Load — Simpan ke Hive Curated

In [ ]:
# TODO: Rename kolom agar sesuai target schema, tambah load_timestamp
spark.sql('CREATE DATABASE IF NOT EXISTS adventureworks_curated')

df_final = df_vendor_summary \
    .withColumn('load_timestamp', F.current_timestamp())

# TODO: Rename kolom agar snake_case dan sesuai spec
# Kemudian simpan ke adventureworks_curated.fact_vendor_performance
# mode: overwrite, saveAsTable

print('=== Verifikasi ===')
df_verify = spark.table('adventureworks_curated.fact_vendor_performance')
print(f'Total rows: {df_verify.count():,}')
df_verify.show(10, truncate=False)

## 7. Analytic Queries

**Wajib: minimal 3 query yang menjawab business question**

In [ ]:
# ── Query 1 (WAJIB): Ranking vendor berdasarkan VendorScore ────────────────
# Tampilkan: vendor_name, ship_method, on_time_rate, avg_price_variance, vendor_score
# Sort: desc vendor_score

print('=== Query 1: Vendor Ranking by VendorScore ===')
spark.sql("""
    -- TODO: tulis query SQL kamu di sini
    SELECT 1
""")


In [ ]:
# ── Query 2 (WAJIB): Perbandingan OnTimeRate per ShipMethod ────────────────
# Tampilkan: ship_method, avg_on_time_rate, total_vendors, total_orders

print('=== Query 2: OnTimeRate per ShipMethod ===')
# TODO

In [ ]:
# ── Query 3 (WAJIB): Vendor dengan AvgPriceVariance negatif ───────────────
# Artinya: beli lebih murah dari StandardCost (menguntungkan perusahaan)
# Tampilkan: vendor_name, ship_method, avg_price_variance, on_time_rate, vendor_score

print('=== Query 3: Vendor dengan Harga Lebih Murah dari StandardCost ===')
# TODO

In [ ]:
# ── Query 4 (BONUS): Korelasi CreditRating dan OnTimeRate ──────────────────
# Tampilkan: credit_rating, avg_on_time_rate, avg_vendor_score, count_vendors
# Insight: apakah vendor dengan credit rating tinggi lebih on-time?

print('=== Query 4 (Bonus): CreditRating vs OnTimeRate ===')
# TODO

In [ ]:
# ── Query 5 (BONUS): Top 3 vendor per ShipMethod berdasarkan TotalOrders ───
# Gunakan window function rank() partitionBy ship_method, orderBy desc total_orders

print('=== Query 5 (Bonus): Top 3 Vendor per ShipMethod by TotalOrders ===')
# TODO

In [ ]:
spark.stop()
print('Done!')